# Edu Task Topic Classifier: от данных к выводам

Исследование классификации русскоязычных задач по математике и физике. Цель — определить предмет и несколько наиболее конкретных тем, сохранив смысл неполной иерархической разметки.

Ноутбук документирует завершённый эксперимент и воспроизводит анализ опубликованных агрегатов. `Run All` работает без исходных данных и весов: длительные действия выключены явно. Чтобы повторить обучение, установите зависимости из README, предоставьте локальные CSV и включите соответствующие флаги. Опубликованные результаты не выдаются за результаты нового запуска.

## 1. Окружение и пути

Код отделён от исследования в пакете `pmc`. Все пути относительны к подпроекту. Выводы ниже опираются на сохранённые отчёты; обучение всегда пишет в новые каталоги.

In [1]:
from pathlib import Path
import csv, json, sys, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pmc").is_dir(), "Запустите notebook из подпроекта или notebooks/"
sys.path.insert(0, str(ROOT))
SOURCE = ROOT / "data/normalized"
DATA = ROOT / "artifacts/notebook_data"
RUN_BASELINE = ROOT / "runs/notebook_baseline"
RUN_BERT = ROOT / "runs/notebook_bert"
DO_PREPARE = False
DO_TRAIN_BASELINE = False
DO_TRAIN_BERT = False
DO_PREDICT = False

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def cli(*args):
    subprocess.run([sys.executable, "-m", "pmc", *map(str, args)], cwd=ROOT, check=True)

print("Документирующий режим: обучение выключено")

Документирующий режим: обучение выключено


## 2. Загрузка и проверка данных

Исходные CSV не распространяются. Требуется таблица задач и дерево тем; подробный контракт находится в `data/README.md`. Ответ и решение нужны только для аудита исходника. Классификатор видит исключительно `task`.

Нормализация запускается отдельно:
```sh
python scripts/normalize_dataset_v2.py --tasks data/raw/tasks.csv --topics data/raw/topics.csv --output-dir data/normalized --seed 42
```
Новый каталог предотвращает случайное обновление результатов ручной проверки. Ячейка проверяет наличие файлов и заголовки, не выводя тексты задач.

In [2]:
required = ["tasks_normalized_v2.csv", "task_topic_links_v2.csv", "manual_review_queue_v2.csv"]
source_ready = all((SOURCE / name).is_file() for name in required)
if source_ready:
    for name in required:
        with (SOURCE / name).open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            print(name, reader.fieldnames, "строк:", sum(1 for _ in reader))
else:
    print("Локальные CSV отсутствуют; далее доступны опубликованные агрегаты.")
summary = read_json(ROOT / "reports/dataset_summary.json")
print("Исходный эксперимент:", summary["split_counts"])
print("Исключено:", summary["excluded_rows"], "Тем:", summary["labels"])

Локальные CSV отсутствуют; далее доступны опубликованные агрегаты.
Исходный эксперимент: {'train': 31811, 'validation': 4002, 'test': 4002}
Исключено: 205 Тем: 663


### Проверенные финальные результаты v2

42 648 исходных строк восстановлены без потерь в 40 020 отдельных задач; сохранены все 42 648 уникальных пар задача–тема. В справочнике 688 тем: использованы 594, не использованы 94, неизвестных ID — 0. Это статистика исходной таксономии, а 663 выхода модели ниже — отдельный словарь узлов путей без корней.

**Cross-subject разметка не обнаружена:** 0 задач с двумя предметами по ID и 0 групп одинакового нормализованного текста с разными предметами. Предмет определяется корнем дерева, а не ключевыми словами. Это не исключает межпредметного содержания или ошибок разметки; несколько тем одной задачи не означают несколько предметов. Метки автоматически не исправлялись.

| Нормализация v2 | Групп | Задач |
|---|---:|---:|
| Семантические группы, всего | 39 776 | 40 020 |
| Одинаковые условия у разных ID | 168 | 336 |
| Действительно различающиеся числовые варианты | 67 | 152 |
| Очередь ручной проверки (`review_group_id`) | 100 | 200 |

Очередь включает все 94 группы с разными темами и все 20 групп с разными непустыми ответами; категории пересекаются и не суммируются. Ответы сравнивались как строки, без проверки математической эквивалентности. 100 групп проверки входят в **98 семантических групп**. Исключение этих семантических групп целиком при подготовке обучения удаляет **205 задач**, все из train. Поэтому размер очереди 200 и число исключённых задач 205 не противоречат друг другу.

Разбиение v2 **до исключения очереди**, seed 42:

| Предмет | Всего | Train | Validation | Test |
|---|---:|---:|---:|---:|
| Математика | 24 997 | 19 997 | 2 500 | 2 500 |
| Физика | 15 023 | 12 019 | 1 502 | 1 502 |
| Всего | 40 020 | 32 016 | 4 002 | 4 002 |

Финальная проверка подтвердила восстановление 42 648 исходных строк, сохранение всех пар задача–тема, отсутствие пересечений семантических групп и точного текста внутри предмета между split, независимость split от порядка строк и побайтовую воспроизводимость повторного запуска. Хэши исходников до и после совпали; исходники и предыдущие файлы не менялись. Это результаты `verification_v2.json`, а не утверждение, что все эти проверки повторно запускались при публикации.

[Компактная сводка v2](../reports/normalization_v2_summary.json) содержит числа и результаты проверок без текстов задач и абсолютных путей. Хэши трёх финальных CSV совпадают с входными хэшами сохранённого обучающего манифеста: обучение уже использовало эти данные v2. Обновление документации не является переобучением и не меняет метрики модели.

Следующие этапы используют подготовленную выборку после исключения 205 задач.



In [3]:
normalization = read_json(ROOT / "reports/normalization_v2_summary.json")
assert sum(normalization["subject_tasks"].values()) == normalization["unique_tasks"] == 40020
assert normalization["cross_subject"] == {"task_ids": 0, "normalized_text_groups": 0}
assert all(normalization["verification_checks"].values())
for subject, splits in normalization["split_by_subject"].items():
    print(subject, splits)
assert sum(x["train"] for x in normalization["split_by_subject"].values()) - normalization["manual_review"]["excluded_after_semantic_expansion"] == summary["split_counts"]["train"]
assert all(normalization["sha256"][name] == value for name, value in summary["source_sha256"].items())
print("Очередь и исключение целых групп:", normalization["manual_review"])
print("Проверки из финального отчёта:", normalization["verification_checks"])
print("Хэши v2 совпадают с входами завершённого обучения.")

Математика {'train': 19997, 'validation': 2500, 'test': 2500}
Физика {'train': 12019, 'validation': 1502, 'test': 1502}
Очередь и исключение целых групп: {'review_groups': 100, 'tasks': 200, 'semantic_groups': 98, 'excluded_after_semantic_expansion': 205, 'excluded_by_split': {'train': 205, 'validation': 0, 'test': 0}, 'automatic_label_corrections': False}
Проверки из финального отчёта: {'lossless_roundtrip': True, 'task_topic_pairs_preserved': True, 'semantic_groups_disjoint': True, 'subject_scoped_exact_text_disjoint': True, 'row_order_independent_split': True, 'byte_identical_rerun': True, 'source_and_previous_files_unchanged_during_rerun': True, 'grouping_boundary_examples': True}
Хэши v2 совпадают с входами завершённого обучения.


## 3. Нормализация, дедупликация и семантические группы

Повторные строки одного ID объединяются без потери исходных записей. Из назначенных тем выбираются наиболее конкретные; соседние ветви сохраняются. NFC и свёртка пробелов применяются к ключу сравнения, а условие остаётся исходным.

Одинаковый текст и допустимые числовые шаблоны объединяются в `semantic_group_id`. Для шаблона нужны минимум 8 буквенных токенов, 50 букв и 1–12 чисел; формулы, разметка, URL и цифры рядом с буквами исключают шаблонную замену. Это консервативная эвристика, а не модель семантического сходства.

Группы целиком распределяются примерно 80/10/10 внутри предметов, seed 42. Темы отдельно не стратифицируются. Поэтому редкая тема может отсутствовать в train. Спорная задача исключает всю свою группу. Следующая ячейка при явном включении создаёт подготовленные артефакты; существующие файлы не перезаписываются.

In [4]:
if DO_PREPARE:
    if not source_ready:
        print("Пропуск: предоставьте три нормализованных CSV.")
    elif DATA.exists():
        print("Пропуск: выберите новый DATA для подготовки.")
    else:
        cli("prepare", "--source", SOURCE, "--output", DATA)
else:
    print("Подготовка выключена; установите DO_PREPARE=True после проверки путей.")

Подготовка выключена; установите DO_PREPARE=True после проверки путей.


## 4. Проверка разбиения без известных утечек

Не делаем случайный split по строкам: варианты одного условия иначе попадут и в обучение, и в оценку. `prepare` проверяет ID, группы и точный нормализованный текст; SHA-256 защищает подготовленные файлы от незаметных изменений. Проверка не исключает нераспознанные перефразировки.

Test здесь не используется для расчёта качества. Для локального аудита ниже проверяются контрольные суммы и пересечение групп train/validation.

In [5]:
from pmc.data import verify_artifacts, load_rows
if (DATA / "manifest.json").exists():
    verify_artifacts(DATA)
    train_rows = load_rows(DATA, "train")
    validation_rows = load_rows(DATA, "validation")
    assert not ({r["group"] for r in train_rows} & {r["group"] for r in validation_rows})
    print("Train/validation: группы не пересекаются, контрольные суммы совпали.")
else:
    print("Локальные артефакты отсутствуют. Проверки выполняются при prepare.")

Локальные артефакты отсутствуют. Проверки выполняются при prepare.


## 5. Почему необходима маска неизвестных меток

Если размечена «Алгебра», отсутствие «Уравнений» не доказывает отрицание. Если размечены «Уравнения», предок не является точной наиболее конкретной темой и также не должен стать отрицательным примером. Проверим это на искусственном дереве без исходных данных.

In [6]:
from pmc.hierarchy import label_state
schema_demo = {"labels": ["2", "3", "4"], "nodes": {
    "1": {"path": ["1"]},
    "2": {"path": ["1", "2"]},
    "3": {"path": ["1", "2", "3"]},
    "4": {"path": ["1", "4"]}}}
for target in ["2", "3"]:
    y, mask = label_state([target], schema_demo)
    print("Цель:", target, "Положительные:", y, "Известные пары:", mask)
assert label_state(["2"], schema_demo) == ([1, 0, 0], [1, 0, 1])
assert label_state(["3"], schema_demo) == ([0, 1, 0], [0, 1, 1])

Цель: 2 Положительные: [1, 0, 0] Известные пары: [1, 0, 1]
Цель: 3 Положительные: [0, 1, 0] Известные пары: [0, 1, 1]


## 6. Baseline: TF-IDF + линейные классификаторы

Слова 1–2 граммы и символы 3–5 грамм дают разреженное представление. SGD с log-loss обучает предмет и отдельные темы. Векторизатор обучается только на train; неизвестные пары исключаются. Это необходимая точка отсчёта для оценки полезности трансформера.

Текущий baseline нужно обучить заново: опубликованный исторический test-результат использовал другую семантику меток и старый формат сохранения. Полный запуск ниже включается отдельно; сначала рекомендуется smoke-команда из README.

In [7]:
if DO_TRAIN_BASELINE:
    if not (DATA / "manifest.json").exists():
        print("Пропуск: сначала подготовьте DATA.")
    elif RUN_BASELINE.exists():
        print("Пропуск: выберите новый RUN_BASELINE.")
    else:
        cli("train", "--data", DATA, "--config", ROOT / "configs/baseline_full.json", "--run", RUN_BASELINE)
else:
    print("Baseline training выключен.")
legacy = read_json(ROOT / "reports/baseline_legacy/test_metrics.json")
print("Исторический test, отдельный протокол:", legacy)

Baseline training выключен.
Исторический test, отдельный протокол: {'n': 4002, 'subject_accuracy': 0.9940029985007496, 'subject_macro_f1': 0.9936019184652278, 'topic_micro_f1': 0.8455580294145316, 'topic_macro_f1': 0.39325033775996154, 'precision@3': 0.8514909212060543, 'recall@3': 0.6622670807453339, 'hierarchical_f1': 0.8451382805656363}


## 7. ruBERT и обучение на MPS

Общий ruBERT-энкодер и две головы обучаются совместно: предмет и masked multi-label темы. Полный запуск на MPS использовал FP32, batch 1, accumulation 16, длину 128, gradient checkpointing и фиксированный padding. Очистка кэша уменьшает давление на память, но лимит аллокатора не гарантирует ограничение всей RAM.

Checkpoint выбирается по validation loss; после выбора весов подбирается общий порог на validation. Test в этих действиях не участвует. Первая загрузка требует сети; MPS-конфигурация требует совместимый Mac. Для другой платформы измените копию конфига. Автоматического продолжения из last.pt нет.

In [8]:
config = read_json(ROOT / "reports/bert/config.json")
print(json.dumps(config["bert"], ensure_ascii=False, indent=2))
if DO_TRAIN_BERT:
    if not (DATA / "manifest.json").exists():
        print("Пропуск: сначала подготовьте DATA.")
    elif RUN_BERT.exists():
        print("Пропуск: выберите новый RUN_BERT.")
    else:
        cli("train", "--data", DATA, "--config", ROOT / "configs/bert_mac_m4_48gb.json", "--run", RUN_BERT)
else:
    print("ruBERT training выключен.")

{
  "model": "ai-forever/ruBert-base",
  "revision": "05f37a2ca9e333fd18f30cd0c96c68d274793c69",
  "device": "mps",
  "batch_size": 1,
  "gradient_accumulation": 16,
  "max_length": 128,
  "epochs": 6,
  "learning_rate": 2e-05,
  "patience": 2,
  "min_delta": 0.0001,
  "pos_weight_cap": 20,
  "topic_loss_weight": 1.0,
  "gradient_checkpointing": true,
  "cpu_threads": 6,
  "fixed_padding": true,
  "optimizer_foreach": false,
  "mps_empty_cache_interval": 16,
  "mps_memory_fraction": 0.65,
  "log_interval": 128
}
ruBERT training выключен.


## 8. Динамика обучения

Загружаем фактическую историю завершённого эксперимента. Снижение validation loss подтверждает улучшение этой целевой функции; оно само по себе не доказывает улучшение каждой редкой темы или качества на другом источнике.

In [9]:
history = read_json(ROOT / "reports/bert/history.json")
print("epoch  train_loss  validation_loss")
for row in history:
    print(f"{row['epoch']:5d}  {row['train_loss']:.6f}    {row['validation_loss']:.6f}")
print("Лучшая эпоха:", min(history, key=lambda r: r["validation_loss"])["epoch"])
print(read_json(ROOT / "reports/bert/training.json"))

epoch  train_loss  validation_loss
    1  0.222722    0.140291
    2  0.107505    0.099068
    3  0.065950    0.082760
    4  0.042467    0.073400
    5  0.029346    0.071826
    6  0.021545    0.064139
Лучшая эпоха: 6
{'device': 'mps', 'precision': 'float32', 'active_topics': 591, 'best_validation_loss': 0.06413916821230063, 'epochs_completed': 6}


## 9. Метрики: точная тема и путь в дереве

Micro-F1 суммирует TP/FP/FN по известным парам; macro-F1 усредняет качество тем. Вариант «все темы» включает нулевые значения для тем без поддержки, вариант «поддержанные» учитывает темы с положительными примерами evaluation. Иерархическая метрика добавляет предков, но исключает корни, чтобы предмет не завышал качество тем.

Порог 0.85 выбран на validation и отражает компромисс precision/recall именно на этой выборке. Эти результаты оптимистичнее независимого test. Сравнивать с историческим baseline напрямую нельзя.

In [10]:
metrics = read_json(ROOT / "reports/bert/validation_metrics.json")
print(f"Subject accuracy: {metrics['subject_accuracy']:.4f}")
for name in ["topics_masked", "hierarchical_masked_without_roots"]:
    print(name)
    for key in ["micro_precision", "micro_recall", "micro_f1", "macro_f1_all_labels", "macro_f1_supported_labels"]:
        print(f"  {key}: {metrics[name][key]:.4f}")
print("Порог:", metrics["threshold"], "Задач:", metrics["n"])

Subject accuracy: 0.9950
topics_masked
  micro_precision: 0.7165
  micro_recall: 0.5984
  micro_f1: 0.6522
  macro_f1_all_labels: 0.3343
  macro_f1_supported_labels: 0.4120
hierarchical_masked_without_roots
  micro_precision: 0.9003
  micro_recall: 0.7359
  micro_f1: 0.8098
  macro_f1_all_labels: 0.4220
  macro_f1_supported_labels: 0.4587
Порог: 0.85 Задач: 4002


## 10. Редкие темы

Группируем по поддержке, зафиксированной в отчёте запуска. Нулевой F1 в группе без train-примеров ожидаем: модель отключает неподдержанные темы. Нулевой F1 при 1–4 примерах показывает, что один высокий общий micro-F1 скрывает важный провал. Per-topic значения при единичных validation-примерах нестабильны; их нельзя трактовать как надёжный рейтинг.

In [11]:
print("Группа | Тем | micro-F1 | recall")
for name, values in metrics["rare_topics"].items():
    print(f"{name} | {values['label_count']} | {values['micro_f1']:.4f} | {values['micro_recall']:.4f}")
print("Примеры тем с train 5–19 и поддержкой validation (по возрастанию F1):")
rare = [r for r in metrics["per_topic"] if 5 <= r["train_support"] < 20 and r["eval_support"] > 0]
for row in sorted(rare, key=lambda r: (r["f1"], r["id"]))[:10]:
    print(row["name"], "train:", row["train_support"], "validation:", row["eval_support"], "F1:", round(row["f1"], 3))

Группа | Тем | micro-F1 | recall
unseen_train | 72 | 0.0000 | 0.0000
train_1_to_4 | 26 | 0.0000 | 0.0000
train_5_to_19 | 162 | 0.1767 | 0.0980
train_20_plus | 403 | 0.6723 | 0.6343
Примеры тем с train 5–19 и поддержкой validation (по возрастанию F1):
Производная произведения и частного функций train: 15 validation: 2 F1: 0.0
коэффициенты многочленов train: 8 validation: 1 F1: 0.0
Формула для xⁿ-yⁿ train: 7 validation: 1 F1: 0.0
угол между стрелками часов train: 5 validation: 1 F1: 0.0
измерения угла с помощью угольника train: 6 validation: 1 F1: 0.0
продление медианы за свою длину train: 9 validation: 1 F1: 0.0
Движение вдогонку и движение с отставанием train: 18 validation: 2 F1: 0.0
преобразование и вычисление алгебраических выражений train: 11 validation: 1 F1: 0.0
Движение тела,брошенного горизонтально train: 10 validation: 1 F1: 0.0
упругий удар train: 6 validation: 1 F1: 0.0


## 11. Предсказание и финальная оценка

Демонстрация использует новое искусственное условие и только локально обученные веса. Численные предсказания без весов не имитируются. Пустой список тем означает, что порог не пройден. Предмет и темы пока не имеют жёсткой взаимной фильтрации.

Финальный test запускайте отдельно после выбора модели командой из README. CLI резервирует test один раз на подготовленный набор, даже если оценка затем завершится ошибкой. Здесь test автоматически не запускается. Исторический baseline test уже известен; дальнейшие выводы должны учитывать этот факт.

In [12]:
if DO_PREDICT and (RUN_BERT / "COMPLETE.json").exists() and (RUN_BERT / "best.pt").exists():
    cli("predict", "--run", RUN_BERT, "--text", "Найдите силу тока при напряжении 12 В и сопротивлении 4 Ом.")
else:
    print("Предсказание пропущено: нужны локальные веса и DO_PREDICT=True.")

Предсказание пропущено: нужны локальные веса и DO_PREDICT=True.


## 12. План улучшений

1. Проверить конфликтные группы и расширить разметку редких тем — это адресует выявленные ошибки данных и недостаток поддержки.
2. Переобучить baseline с текущими масками и зафиксировать общий протокол сравнения. Без этого нельзя утверждать преимущество ruBERT.
3. Измерить обрезание текста на 128 токенах, затем сравнить 256/512 с учётом памяти. Отдельно исследовать задачи с формулами и изображениями.
4. Проверить согласованность предмета и темы, калибровку, долю отказов и качество при разных порогах. Порог подбирать только на validation.
5. Исследовать новые источники, перефразировки, доверительные интервалы и ошибки по группам, прежде чем внедрять автоматическую разметку.

## Большой итоговый вывод

Проект показывает полный путь от неоднородной таблицы учебных задач до работающего многозадачного классификатора. Существенная часть результата — проверяемая обработка данных: сохранение исходных записей, выделение наиболее конкретных меток, группировка повторов и контроль разбиения. Эти этапы делают оценку осмысленной: модель должна узнавать содержание задачи, а не повторять близкий пример из train или читать ответ и название темы среди признаков.

Завершённый ruBERT-эксперимент подтверждает возможность обучения на потребительском Apple Silicon: шесть эпох на MPS, сохранение лучших весов и автоматический подбор общего порога. Subject accuracy 0.9950 на validation говорит о хорошо решаемой грубой классификации. Exact masked micro-F1 0.6522 и hierarchical micro-F1 0.8098 описывают более сложную картину: правильная область учебной программы определяется легче, чем конкретная тема. Более высокая иерархическая метрика не заменяет точную и не доказывает готовность детальной автоматической разметки.

Анализ поддержки раскрывает главный практический предел. Для тем с 5–19 train-примерами micro-F1 около 0.1767, а для 1–4 примеров равен нулю. Усреднение по всем темам даёт macro-F1 0.3343. Поэтому систему разумно рассматривать как помощника преподавателя: она предлагает темы и может отказаться от ответа, но редкие случаи требуют проверки. Эти выводы опираются на агрегаты; причины конкретных ошибок потребуют локального просмотра условий и разметки.

Исторический baseline полезен как след развития проекта, однако его test topic micro-F1 0.8456 измеряет другую семантику. Он не доказывает ни превосходства baseline, ни отставания ruBERT. Честный следующий эксперимент — единые маски, словарь, split, процедура выбора порога и заранее оговорённая итоговая оценка. Поскольку ruBERT validation использовалась для выбора эпохи и порога, её качество нельзя выдавать за независимый test-результат.

Главный инженерный вывод: улучшение классификатора начинается с разметки, протокола и анализа редких тем. Новая архитектура может помочь, но без этих оснований рост одной цифры мало говорит о практической полезности. Публичный подпроект сохраняет код и доказуемые результаты, позволяет изучить исследование без приватного корпуса и задаёт прозрачный путь к следующему этапу.

Финальная нормализация v2 подтверждает сохранность 42 648 исходных строк и 40 020 задач. Cross-subject меток не обнаружено: многометочность относится к темам внутри одного предмета. Различие между 200 задачами в очереди и 205 исключёнными объясняется расширением до целых семантических групп. Хэши v2 совпадают с обучающими входами, поэтому новые отчёты уточняют происхождение и проверку данных, а не представляют новый запуск модели.
